# Screen Macro — Object Detection Training

Trains a small object-detection model on a dataset captured/labeled by Screen Macro's "Train new model…" wizard (Detect Object action), and exports it to the `.onnx` format the app's `onnx_detector.py` expects.

**Colab (default, manual):**
1. `Runtime` → `Change runtime type` → select a **GPU** (T4 is fine, free tier).
2. `Runtime` → `Run all`.
3. When prompted, upload the `..._dataset.zip` file Screen Macro produced (from the action's "Train new model…" button).
4. When training finishes, `best.onnx` downloads automatically — import it back into the same action via "Import model…".

**Kaggle (opt-in, scripted):** the app's `cv_training.py` pushes this notebook as a Kaggle kernel with the dataset attached as a Kaggle Dataset, polls until it finishes, and pulls `best.onnx` back automatically — no manual steps. The notebook auto-detects which platform it's running on (`ON_KAGGLE`, cell 2) and adjusts the upload/download cells accordingly; everything else runs identically either way.

**Status: first draft, not yet run end-to-end.** The dataset-conversion and export-contract cells are straightforward and should just work. The actual RF-DETR training/export API calls (cells marked ⚠️ below) are written from documentation, not a verified hands-on run — if a cell errors, the fix is almost always a small argument-name/shape mismatch specific to whatever `rfdetr` version installs, not a design problem with the pipeline itself. See `docs/cv-object-detection-investigation.md` in the main app repo for the background.

## 1. Install dependencies

In [ ]:
!pip install -q rfdetr onnx onnxruntime supervision

## 2. Upload and unpack the dataset

Expects the zip Screen Macro's "Train new model…" wizard produces: an `images/` folder of captured frames plus a `labels.json` of the form
`{"images_dir": "images", "labels": [{"image": "frame_001.png", "box": [x1, y1, x2, y2], "keypoint": [x, y]}, ...]}`
(frames with no entry in `labels` were skipped during labeling — the object wasn't visible in them).

In [ ]:
import os, glob, zipfile, shutil
from pathlib import Path

RAW_DIR = Path("dataset_raw")
if RAW_DIR.exists():
    shutil.rmtree(RAW_DIR)

ON_KAGGLE = os.path.exists("/kaggle/input")
if ON_KAGGLE:
    # cv_training.py's push already uploaded the dataset zip as this kernel's one
    # attached Kaggle Dataset -- mounted read-only under /kaggle/input, no interactive
    # upload prompt needed the way Colab's files.upload() requires.
    candidates = glob.glob("/kaggle/input/*/*.zip") + glob.glob("/kaggle/input/*.zip")
    zip_name = candidates[0]
else:
    from google.colab import files
    uploaded = files.upload()
    zip_name = next(iter(uploaded))

with zipfile.ZipFile(zip_name) as zf:
    zf.extractall(RAW_DIR)

print("Extracted:", list(RAW_DIR.iterdir()))

## 3. Convert to COCO format (train/valid split)

One category (`"object"`) with one keypoint (`"click_point"`) per COCO's keypoint-annotation convention — `keypoints: [x, y, visibility]`, `visibility=2` meaning "labeled and visible" (every entry here is, since the labeling step only records a keypoint when the object was actually clicked on). A 90/10 split is a reasonable default for a small, single-object dataset; adjust `VALID_FRACTION` if you captured a lot more images.

In [ ]:
import json, random

VALID_FRACTION = 0.1
COCO_DIR = Path("dataset_coco")

with open(RAW_DIR / "labels.json") as f:
    raw = json.load(f)
images_dir = RAW_DIR / raw.get("images_dir", "images")
entries = raw["labels"]
random.seed(0)
random.shuffle(entries)
n_valid = max(1, int(len(entries) * VALID_FRACTION)) if len(entries) > 1 else 0
splits = {"valid": entries[:n_valid], "train": entries[n_valid:]}

import cv2

for split_name, split_entries in splits.items():
    split_dir = COCO_DIR / split_name
    split_dir.mkdir(parents=True, exist_ok=True)
    images_out, annotations_out = [], []
    for i, e in enumerate(split_entries, start=1):
        src = images_dir / e["image"]
        img = cv2.imread(str(src))
        h, w = img.shape[:2]
        shutil.copy(src, split_dir / e["image"])
        x1, y1, x2, y2 = e["box"]
        kx, ky = e["keypoint"]
        images_out.append({"id": i, "file_name": e["image"], "width": w, "height": h})
        annotations_out.append({
            "id": i, "image_id": i, "category_id": 1,
            "bbox": [x1, y1, x2 - x1, y2 - y1],
            "area": (x2 - x1) * (y2 - y1),
            "iscrowd": 0,
            "keypoints": [kx, ky, 2],
            "num_keypoints": 1,
        })
    coco = {
        "images": images_out,
        "annotations": annotations_out,
        "categories": [{
            "id": 1, "name": "object", "supercategory": "object",
            "keypoints": ["click_point"], "skeleton": [],
        }],
    }
    with open(split_dir / "_annotations.coco.json", "w") as f:
        json.dump(coco, f)
    print(f"{split_name}: {len(images_out)} image(s)")

## 4. Train ⚠️ (unverified API)

Tries RF-DETR's keypoint-preview model first (predicts the click point directly, per `docs/cv-object-detection-investigation.md`'s recommendation); falls back to the plain box-only model if the preview class isn't importable or training on it fails — a fully-supported degradation, not an error path (`onnx_detector.py` already handles a keypoint-less `[N,6]` export by clicking the box center instead).

In [ ]:
KEYPOINT_MODE = True
try:
    from rfdetr import RFDETRKeypointPreview
    model = RFDETRKeypointPreview()
except Exception as exc:
    print("Keypoint-preview model unavailable, falling back to box-only:", exc)
    KEYPOINT_MODE = False

if not KEYPOINT_MODE:
    from rfdetr import RFDETRNano
    model = RFDETRNano()

EPOCHS = 50       # small dataset -- more epochs than a COCO-scale run, adjust if it overfits
BATCH_SIZE = 4    # keep small: free-tier T4 memory headroom is the main constraint here

try:
    model.train(dataset_dir=str(COCO_DIR), epochs=EPOCHS, batch_size=BATCH_SIZE)
except Exception as exc:
    if KEYPOINT_MODE:
        print("Keypoint-preview training failed, falling back to box-only:", exc)
        from rfdetr import RFDETRNano
        KEYPOINT_MODE = False
        model = RFDETRNano()
        model.train(dataset_dir=str(COCO_DIR), epochs=EPOCHS, batch_size=BATCH_SIZE)
    else:
        raise

## 5. Export to ONNX with the app's expected contract ⚠️ (unverified API)

`onnx_detector.py` expects a single output tensor per image: `[N, 6]` (x1, y1, x2, y2, confidence, class_id) or `[N, 8]` (…+keypoint_x, keypoint_y) in the model's own fixed input-size (letterboxed) pixel coordinates — with NMS/decoding already applied, so the shipped app never needs architecture-specific postprocessing. This wrapper runs the trained model's own `predict()` (which already does that decoding internally) inside a `forward()`, then concatenates the result into that flat contract before `torch.onnx.export`.

In [ ]:
import torch
import torch.nn as nn

CONF_THRESHOLD = 0.3
INPUT_SIZE = 640

class ExportWrapper(nn.Module):
    """Wraps a trained RF-DETR model's own predict() (already does NMS/decoding
    internally) so ONNX export produces one flat [N,6]/[N,8] tensor per image instead of
    RF-DETR's native per-field outputs -- this is the piece that lets onnx_detector.py
    stay completely architecture-agnostic."""

    def __init__(self, base_model, keypoint_mode: bool, conf_threshold: float):
        super().__init__()
        self.base_model = base_model
        self.keypoint_mode = keypoint_mode
        self.conf_threshold = conf_threshold

    def forward(self, x):
        detections = self.base_model.predict(x, threshold=self.conf_threshold)
        boxes = detections.xyxy            # (N, 4)
        scores = detections.confidence      # (N,)
        classes = detections.class_id       # (N,)
        cols = [boxes, scores.unsqueeze(-1), classes.unsqueeze(-1).float()]
        if self.keypoint_mode and getattr(detections, "keypoints", None) is not None:
            cols.append(detections.keypoints.reshape(detections.keypoints.shape[0], -1)[:, :2])
        return torch.cat(cols, dim=-1)

wrapper = ExportWrapper(model, KEYPOINT_MODE, CONF_THRESHOLD)
wrapper.eval()
dummy_input = torch.zeros(1, 3, INPUT_SIZE, INPUT_SIZE)
torch.onnx.export(
    wrapper, dummy_input, "best.onnx",
    input_names=["images"], output_names=["output0"],
    dynamic_axes={"output0": {0: "num_detections"}},
    opset_version=17,
)
print("Exported best.onnx (keypoint_mode=", KEYPOINT_MODE, ")")

## 6. Download the trained model

In [ ]:
if ON_KAGGLE:
    # Kaggle captures every file left in /kaggle/working/ as this kernel's output --
    # no equivalent of Colab's interactive download; cv_training.py's poll/pull step
    # fetches it afterward via `kaggle kernels output`.
    print("On Kaggle: best.onnx left in /kaggle/working/ -- fetched by the app's Kaggle poll/pull step.")
else:
    from google.colab import files
    files.download("best.onnx")